<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2014%20-%202026-05-12%20-%20Hip%C3%B3tesis%20y%20pruebas%20estad%C3%ADsticas/clase_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 14 · Hipótesis y pruebas estadísticas

Ayer vimos cómo visualizar datos. Hoy: ¿cómo sé si lo que veo es realmente ahí o solo ruido? Es lo que preguntarán en cualquier entrevista de datos. Y la mayoría de la gente lo responde mal.

Un p-valor no es "la probabilidad de que mi hipótesis sea cierta". Si lo dices así, la entrevista terminó. Es "la probabilidad de ver estos datos si la hipótesis nula fuera verdadera".

Hoy ejecutamos t-tests, chi-cuadrado, ANOVA. Y aprendemos a leer p-valores sin autoengañarse.

> **Hoy haces** · Los tres ejercicios de hipótesis y pruebas (t-test, chi-cuadrado, p-values), las dos pausas y el checkpoint. ~2h.
>
> **Entrega** · El notebook ejecutado con la interpretación de cada test, al repo del equipo antes de la próxima clase.

## Setup

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scipy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"pandas {pd.__version__} · scipy {scipy.__version__}")

---

## La pregunta real

Intuición: "Primero las mujeres y los niños" en el Titanic. ¿Fue cierto? ¿Las mujeres eran más jóvenes en promedio que los hombres?

Miro un gráfico y veo que sí, pareciera. Pero ¿cuán seguro estoy? ¿Es diferencia de verdad o es solo casualidad por cómo salió la muestra?

In [ ]:
titanic = sns.load_dataset("titanic").dropna(subset=["age"])

print(f"Shape: {titanic.shape}")
print(f"\nEdad por sexo:")
print(titanic.groupby("sex")["age"].describe())

plt.figure(figsize=(10, 4))
sns.violinplot(data=titanic, x="sex", y="age", palette="Set2")
plt.title("Distribución de edades por sexo (Titanic)")
plt.ylabel("Edad (años)")
plt.tight_layout()

Mirá los dos violines. ¿Son iguales o distintos? A ojo, dirías que sí hay diferencia. Pero... ¿cuánta diferencia es significativa?

Eso es lo que un test estadístico responde.

---

## 1. t-test · comparar dos grupos

El t-test compara las medias de dos grupos y te dice: "¿Esa diferencia es real o era esperable por casualidad?". Devuelve un p-valor. Si p < 0.05 (la convención), rechazas la hipótesis nula ("no hay diferencia").

In [ ]:
edad_hombres = titanic.loc[titanic["sex"] == "male", "age"]
edad_mujeres = titanic.loc[titanic["sex"] == "female", "age"]

print(f"Hombres: n={len(edad_hombres)}, media={edad_hombres.mean():.2f}")
print(f"Mujeres: n={len(edad_mujeres)}, media={edad_mujeres.mean():.2f}")
print(f"Diferencia: {abs(edad_hombres.mean() - edad_mujeres.mean()):.2f} años")

t_stat, p_value = stats.ttest_ind(edad_hombres, edad_mujeres, equal_var=False)

print(f"\nt-statístic = {t_stat:.3f}")
print(f"p-valor = {p_value:.4f}")
print(f"\nInterpretación:")
if p_value < 0.05:
    print("✓ Rechazamos H₀: SÍ hay diferencia significativa en edad entre sexos.")
else:
    print("✗ No rechazamos H₀: No hay evidencia de diferencia significativa.")

Ese p-valor te dice: "Si los hombres y mujeres tuvieran exactamente la misma edad promedio, y yo sacara muestras al azar, ¿qué tan raro sería ver una diferencia tan grande como esta?". Un p de 0.001 significa: muy raro. Uno de 0.4 significa: bastante normal incluso sin diferencia real.

### Ejercicio 1

Usa `tips` (mesadas) y compara la propina promedio entre meseros (`Male`) y meseras (`Female`). Ejecuta un t-test e interpreta.

In [ ]:
tips = sns.load_dataset("tips")

propina_male = tips.loc[tips["sex"] == "___", "tip"]
propina_female = tips.loc[tips["sex"] == "___", "tip"]

t_stat, p_val = stats.ttest_ind(propina_male, propina_female, equal_var=False)

print(f"Propina promedio - Meseros: {propina_male.mean():.2f}, Meseras: {propina_female.mean():.2f}")
print(f"t = {t_stat:.3f}, p = {p_val:.4f}")
print(f"Conclusión: ", end="")
if p_val < 0.05:
    print("Diferencia significativa ✓")
else:
    print("Sin diferencia significativa ✗")

In [ ]:
assert len(propina_male) > 0 and len(propina_female) > 0, "Valores vacíos"
assert isinstance(t_stat, (float, np.floating)), "t_stat debe ser número"
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution]
# propina_male = tips.loc[tips["sex"] == "Male", "tip"]
# propina_female = tips.loc[tips["sex"] == "Female", "tip"]
# t_stat, p_val = stats.ttest_ind(propina_male, propina_female, equal_var=False)

---

## 2. Chi-cuadrado · variables categóricas

Cuando ambas variables son categóricas (no numéricas), el t-test no funciona. Usas chi-cuadrado. Te dice: "¿Estas dos variables categóricas están relacionadas o son independientes?"

In [ ]:
contingency_table = pd.crosstab(titanic["sex"], titanic["survived"])
print("Tabla de contingencia:")
print(contingency_table)
print()

chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print(f"chi² = {chi2:.2f}")
print(f"p-valor = {p_val:.4e}")
print(f"grados de libertad = {dof}")
print(f"\nFrecuencias esperadas (bajo independencia):")
print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns))

if p_val < 0.05:
    print("\n✓ Sexo y supervivencia ESTÁN asociados (dependencia).")
else:
    print("\n✗ No hay evidencia de asociación.")

---

## 3. ANOVA · 3 o más grupos

t-test compara 2 grupos. Si tienes 3+ grupos (ej: 3 clases del Titanic), usas ANOVA. Es como una extensión del t-test para múltiples grupos.

In [ ]:
tarifa_clase1 = titanic.loc[titanic["pclass"] == 1, "fare"].dropna()
tarifa_clase2 = titanic.loc[titanic["pclass"] == 2, "fare"].dropna()
tarifa_clase3 = titanic.loc[titanic["pclass"] == 3, "fare"].dropna()

print(f"Clase 1: media={tarifa_clase1.mean():.2f}, n={len(tarifa_clase1)}")
print(f"Clase 2: media={tarifa_clase2.mean():.2f}, n={len(tarifa_clase2)}")
print(f"Clase 3: media={tarifa_clase3.mean():.2f}, n={len(tarifa_clase3)}")

f_stat, p_val = stats.f_oneway(tarifa_clase1, tarifa_clase2, tarifa_clase3)

print(f"\nF-statístic = {f_stat:.2f}")
print(f"p-valor = {p_val:.4e}")

if p_val < 0.05:
    print("\n✓ Hay diferencia significativa en tarifa entre clases.")
else:
    print("\n✗ Sin diferencia significativa.")

### Ejercicio 2

Compara el total facturado (`total_bill`) de `tips` entre 4 días (Thurs, Fri, Sat, Sun) con ANOVA. ¿Hay diferencia significativa?

In [ ]:
bill_thurs = tips.loc[tips["day"] == "___", "total_bill"]
bill_fri = tips.loc[tips["day"] == "___", "total_bill"]
bill_sat = tips.loc[tips["day"] == "___", "total_bill"]
bill_sun = tips.loc[tips["day"] == "___", "total_bill"]

f_stat, p_val = stats.f_oneway(bill_thurs, bill_fri, bill_sat, bill_sun)

print(f"F = {f_stat:.2f}, p = {p_val:.4f}")
print("Conclusión: ", end="")
if p_val < 0.05:
    print("Diferencia significativa en gasto por día ✓")
else:
    print("Sin diferencia ✗")

print(f"\nPromedio por día:")
print(tips.groupby("day")["total_bill"].mean().sort_values(ascending=False))

In [ ]:
assert len(bill_thurs) > 0 and len(bill_fri) > 0
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# bill_thurs = tips.loc[tips["day"] == "Thurs", "total_bill"]
# bill_fri = tips.loc[tips["day"] == "Fri", "total_bill"]
# bill_sat = tips.loc[tips["day"] == "Sat", "total_bill"]
# bill_sun = tips.loc[tips["day"] == "Sun", "total_bill"]

---

## 4. Tamaño del efecto · más allá del p-valor

El p-valor te dice "¿hay diferencia?". Pero ¿cuán grande es esa diferencia? Un p-valor bajito con diferencia minúscula es técnicamente "significativo" pero prácticamente inútil.

Cohen's d es la diferencia de medias dividida por la desviación estándar. Más fácil de interpretar: d=0.2 (pequeño), d=0.5 (mediano), d=0.8 (grande).

In [ ]:
def cohens_d(x, y):
    """
    Calcula Cohen's d para dos muestras.
    """
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    s = np.sqrt(((nx - 1) * x.var() + (ny - 1) * y.var()) / dof)
    d = (x.mean() - y.mean()) / s
    return d

d = cohens_d(edad_hombres, edad_mujeres)
print(f"Cohen's d (edad: hombres vs mujeres) = {d:.3f}")
print(f"Interpretación: ", end="")
if abs(d) < 0.2:
    print("insignificante")
elif abs(d) < 0.5:
    print("pequeño")
elif abs(d) < 0.8:
    print("mediano")
else:
    print("grande")

### Ejercicio 3

Calcula Cohen's d para la comparación de propinas (meseros vs meseras) del Ejercicio 1.

In [ ]:
d_tips = cohens_d(propina_male, propina_female)
print(f"Cohen's d = {d_tips:.3f}")
print(f"Magnitud: ", end="")
if abs(d_tips) < 0.2:
    print("insignificante")
elif abs(d_tips) < 0.5:
    print("pequeño")
elif abs(d_tips) < 0.8:
    print("mediano")
else:
    print("grande")

In [ ]:
assert isinstance(d_tips, (float, np.floating))
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# d_tips = cohens_d(propina_male, propina_female)

---

## Desafío · Protocolo completo

Diseña y ejecuta un **protocolo completo de prueba de hipótesis** sobre un dataset a tu elección. Debe incluir:

1. Formulación clara de H₀ y H₁ (en palabras, no en símbolos).
2. Selección del test apropiado (¿t-test, chi², ANOVA?) con justificación.
3. Verificación de supuestos (normalidad, homogeneidad de varianzas).
4. Ejecución del test: p-valor + tamaño del efecto.
5. Conclusión en lenguaje de negocio: ¿qué significa este resultado?

Escribe todo en prosa, no solo números.

In [ ]:
# Paso 1: Formulación
# H₀: [escribe aquí]
# H₁: [escribe aquí]

# Paso 2: Preparación de datos
# ← tu código aquí

# Paso 3: Verificación de supuestos
# ← tu código aquí

# Paso 4: Test
# ← tu código aquí

# Paso 5: Conclusión
# ← escribe en 3-4 oraciones

print("\n🎯 Desafío: Entregado")

---

## Lo que aprendiste

t-test para 2 grupos. Chi-cuadrado para categóricas. ANOVA para 3+. p-valor es condicional ("si H₀ fuera cierto..."), no directo. Tamaño del efecto es tan importante como significancia.

Mañana entramos en producto analítico: dashboards interactivos con Streamlit (Día 15).

> **Recordatorio** · Notebook ejecutado al repo antes de la próxima sesión.

---

## Trabajo en equipo

Para el proyecto de ustedes:

- Formulen 2 hipótesis sobre vuestro dataset.
- Ejecuten el test apropiado (t-test, chi², ANOVA).
- Reporten p-valor + tamaño del efecto.

Entrega: `hypothesis_tests.ipynb` con los 2 tests + conclusiones en prosa (5-7 oraciones cada una).